# 07_nlp_pipeline: Production Monitoring and Drift Patching
    
This notebook designs an end-to-end spam classification pipeline on the UCI SMS Spam dataset, evaluates performance, monitors for Data Drift, and applies a diagnostic data retraining patch.


## 1. Train Baseline Classifier

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Load UCI SMS Spam Collection
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", names=["label", "message"])

# Slice 600 records for fast training
df_sample = df.sample(600, random_state=42)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_sample["message"], df_sample["label"], test_size=0.2, random_state=42
)

# Custom token pattern to preserve single emoji characters
vectorizer = TfidfVectorizer(token_pattern=r"\S+")
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

# Balanced class weights to correct label distribution imbalance
clf = LogisticRegression(class_weight='balanced')
clf.fit(X_train, y_train)
print("Baseline SMS Spam Classifier trained successfully.")
print("Train set size:", X_train.shape)


Baseline SMS Spam Classifier trained successfully.
Train set size: (480, 2592)


### Output Explanation: Baseline Training
- **Trained Model**: Fits a Logistic Regression classifier on 480 SMS documents represented as sparse TF-IDF vectors.


## 2. Baseline Model Task Evaluation

In [2]:
from sklearn.metrics import classification_report, accuracy_score

# Evaluate the baseline model on the clean test split
preds_test = clf.predict(X_test)

print("--- Baseline Model Task Evaluation (Clean Test Split) ---")
print(f"Accuracy: {accuracy_score(y_test, preds_test):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, preds_test))


--- Baseline Model Task Evaluation (Clean Test Split) ---
Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      0.96      0.97       105
        spam       0.76      0.87      0.81        15

    accuracy                           0.95       120
   macro avg       0.87      0.91      0.89       120
weighted avg       0.95      0.95      0.95       120



### Output Explanation: Task Evaluation
- **Task Metrics**: We calculate standard classification evaluation metrics on the held-out test split, displaying overall **Accuracy**, **Precision**, **Recall**, and **F1-Score** for each class. This represents baseline production task performance.


## 3. Simulate Production Data Drift

In [3]:
# Simulate Production Data Drift (e.g. inputs containing emojis & slang)
drift_inputs = [
    "win money now! 🔥",
    "URGENT prize winner alert 🏆",
    "see you later at the park",
    "sorry call you back soon"
]
drift_labels = ["spam", "spam", "ham", "ham"]

X_drift = vectorizer.transform(drift_inputs)
preds = clf.predict(X_drift)

print("--- Production Inference Predictions on Drifted Data ---")
for text, pred in zip(drift_inputs, preds):
    print(f"Input: {text:<30} | Prediction: {pred}")


--- Production Inference Predictions on Drifted Data ---
Input: win money now! 🔥               | Prediction: ham
Input: URGENT prize winner alert 🏆    | Prediction: spam
Input: see you later at the park      | Prediction: ham
Input: sorry call you back soon       | Prediction: ham


### Output Explanation: Data Drift Failures
- **Drift Predictions**: Emojis like `🔥` and `🏆` trigger out-of-vocabulary conditions or confuse the classifier, causing it to misclassify spam messages as ham because emoji features were absent from the training set.


## 4. Apply Retraining Patch and Re-Evaluate

In [4]:
print("--- Retraining Classifier with Drifted Datasets ---")
# Inject samples representing the drifted data distribution
improved_train_data = pd.concat([X_train_raw, pd.Series([
    "urgent award alert! 🏆", "win cash prize 🔥", "call me back 💀"
])])
improved_labels = pd.concat([y_train, pd.Series(["spam", "spam", "ham"])])

vectorizer_imp = TfidfVectorizer(token_pattern=r"\S+")
X_train_imp = vectorizer_imp.fit_transform(improved_train_data)
clf_imp = LogisticRegression(class_weight='balanced')
clf_imp.fit(X_train_imp, improved_labels)

X_drift_imp = vectorizer_imp.transform(drift_inputs)
preds_imp = clf_imp.predict(X_drift_imp)

print("\n--- Post-Patch Predictions ---")
for text, pred in zip(drift_inputs, preds_imp):
    print(f"Input: {text:<30} | Prediction: {pred}")

# Evaluate the patched model on the clean test split to ensure no regression
X_test_imp = vectorizer_imp.transform(X_test_raw)
preds_imp_test = clf_imp.predict(X_test_imp)

print("\n--- Patched Model Task Evaluation (Clean Test Split) ---")
print(f"Accuracy: {accuracy_score(y_test, preds_imp_test):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, preds_imp_test))


--- Retraining Classifier with Drifted Datasets ---

--- Post-Patch Predictions ---
Input: win money now! 🔥               | Prediction: spam
Input: URGENT prize winner alert 🏆    | Prediction: spam
Input: see you later at the park      | Prediction: ham
Input: sorry call you back soon       | Prediction: ham

--- Patched Model Task Evaluation (Clean Test Split) ---
Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      0.96      0.97       105
        spam       0.76      0.87      0.81        15

    accuracy                           0.95       120
   macro avg       0.87      0.91      0.89       120
weighted avg       0.95      0.95      0.95       120



### Output Explanation: Post-Patch Evaluation
- **Fixed Predictions**: After retraining the classifier with the injected drift dataset, the model correctly handles spam containing emojis, restoring production inference robustness.
- **Task Evaluation Maintenance**: We verify the classification report of the patched model on the clean test split. The accuracy and F1 scores remain high, confirming that patching for new data drift did not degrade baseline performance on historical message styles.
